# B-02: Deep-learning forecasters (FC-02, D-38)

Native sequence-to-sequence, driver-conditional, multi-horizon, partial-pooled — hand-rolled pure PyTorch
(`src/models/forecasting_dl.py`). Same data (External SMS/MET), CV and eval points as FC-01.

- **DLinear** (Zeng-2023 decomposition-linear, the "simple beats complex" baseline) + future-exog term.
- **LSTM** seq2seq (encoder over CH4+drivers history; decoder over known-future drivers).
- **LSTM+VSN** (variable-selection gate → native variable importance, used in I-01).

Encoder sees [gap-filled CH4, drivers, lagged FCO2/GPP/Reco]; decoder sees ONLY known-future drivers (leak-free).
Metrics on **observed** targets: RMSE/MAE/R²/MBE + skill vs persistence & climatology, per horizon × track × tower.
GPU (RTX 5070) if available, else CPU.

In [1]:
import sys, time, datetime; sys.path.insert(0, "../../src/models")
import numpy as np, pandas as pd, torch
import forecasting_dl as fdl
RESULTS = __import__("pathlib").Path("../../results")
dev = fdl.get_device(); print("device:", dev, "| torch", torch.__version__)
m = fdl.load_matrix(); print("matrix", m.shape, "| FX", len(fdl.FX))
W = {"A": fdl.build_windows(m, "A"), "B": fdl.build_windows(m, "B")}
print("windows: A T4", W["A"][4]["enc"].shape, "| B T4", W["B"][4]["enc"].shape)

device: cuda | torch 2.11.0+cu128


matrix (210459, 48) | FX 19


windows: A T4 (11657, 168, 23) | B T4 (2883, 28, 23)


## 1  Train + evaluate all models × tracks (30 epochs)

In [2]:
MODELS = ["DLinear", "LSTM", "LSTM_VSN"]; EPOCHS = 30
ALL = []
for mdl in MODELS:
    for trk in ["A", "B"]:
        t0 = time.time()
        rows = fdl.run_track(trk, mdl, W, dev, epochs=EPOCHS)
        ALL += rows
        print(f"{mdl:9s} track {trk}: {len(rows):2d} rows  ({time.time()-t0:.0f}s)", flush=True)
R = pd.DataFrame(ALL); print("total rows", len(R))

DLinear   track A: 15 rows  (58s)


DLinear   track B: 12 rows  (12s)


LSTM      track A: 15 rows  (78s)


LSTM      track B: 12 rows  (16s)


LSTM_VSN  track A: 15 rows  (96s)


LSTM_VSN  track B: 12 rows  (18s)


total rows 81


## 2  Results — DL skill vs persistence, and vs FC-01 (RF)

In [3]:
print("=== Track A (hourly) — skill vs persistence ===")
for t in [4,9,2]:
    sub=R[(R.track=="A")&(R.tower==t)]
    print(f"\n-- Tower {t} --"); print(sub.pivot_table(index="model",columns="horizon",values="skill_persist").round(3).to_string())
print("\n=== Track B (daily) — skill vs persistence ===")
for t in [4,9,2]:
    sub=R[(R.track=="B")&(R.tower==t)]
    print(f"\n-- Tower {t} --"); print(sub.pivot_table(index="model",columns="horizon",values="skill_persist").round(3).to_string())
print("\n=== R2 (towers 4/9) ===")
print(R[R.tower.isin([4,9])].pivot_table(index=["track","tower","model"],columns="horizon",values="R2").round(3).to_string())
# compare to FC-01 RF (skill_persist), Track A
try:
    fc1=pd.read_csv(RESULTS/"fc01_summary.csv"); rf=fc1[(fc1.model=="RF")&(fc1.track=="A")]
    print("\n=== FC-01 RF skill_persist (Track A) for reference ===")
    print(rf.pivot_table(index="tower",columns="horizon",values="skill_persist").round(3).to_string())
except Exception as e: print("FC-01 compare skipped:",e)

=== Track A (hourly) — skill vs persistence ===

-- Tower 4 --
horizon      1      6      12     24     48
model                                      
DLinear  -0.044  0.057 -0.106  0.004  0.039
LSTM     -0.151  0.148 -0.147  0.038  0.032
LSTM_VSN -0.102  0.164  0.012  0.053  0.069

-- Tower 9 --
horizon      1      6      12     24     48
model                                      
DLinear   0.002  0.139  0.128  0.134  0.140
LSTM     -0.186  0.186  0.191  0.137  0.128
LSTM_VSN -0.024  0.120  0.187  0.177  0.163

-- Tower 2 --
horizon      1      6      12     24     48
model                                      
DLinear   0.096 -0.095  0.008  0.133  0.168
LSTM      0.375  0.053  0.159  0.254  0.283
LSTM_VSN  0.393  0.047  0.139  0.213  0.247

=== Track B (daily) — skill vs persistence ===

-- Tower 4 --
horizon      1      3      7      14
model                               
DLinear  -0.092  0.040  0.190  0.330
LSTM     -0.376 -0.159  0.043  0.245
LSTM_VSN -0.475 -0.217 -0.071  0.205

## 3  Append to benchmarks.csv (FC-02) + summary

In [4]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="FC-02"]
rows=[]
for _,r in R.iterrows():
    rows.append({"replication":"FC-02","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":"seq2seq driver-conditional (EXT/SMS_MET)","track":r["track"],"horizon":int(r["horizon"]),
        "split":"fc_main" if r.tower in (4,9) else "fc_t2folds","R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],
        "MBE":r["MBE"],"n_test":int(r["n_test"]),"skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],
        "date":today,"notes":"FC-02 deep learning (DLinear/LSTM/LSTM-VSN); native seq2seq; GPU (D-38)"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
R.to_csv(RESULTS/"fc02_summary.csv",index=False)
print(f"Wrote {len(new)} FC-02 rows. Total {len(comb)}. Saved fc02_summary.csv")

Wrote 81 FC-02 rows. Total 3044. Saved fc02_summary.csv
